# Amazon S3 / Azure Blob Storage (Enterprise AI System Design)

Object storage is one of the most important components in any **Enterprise AI / RAG** application because LLMs don't read files directly—they process documents that are stored in object storage.

---

# 1. What is Object Storage?

## Definition

Object Storage is a service used to store **unstructured data** such as:

- PDFs
- Word Documents
- Images
- Audio
- Video
- CSV
- Excel
- JSON

Unlike PostgreSQL, it does not store data in tables.

---

## Interview Answer

> "Amazon S3 and Azure Blob Storage are highly scalable object storage services used to store large unstructured files. In enterprise AI applications, they store uploaded documents that are later processed for OCR, chunking, embedding generation, and indexing into a vector database."

---

# Azure vs AWS

| Azure | AWS |
|--------|-----|
| Azure Blob Storage | Amazon S3 |

---

# 2. Why Do We Need Object Storage?

Suppose a user uploads

```text
HR_Policy.pdf
```

Should we store it inside PostgreSQL?

**No.**

Should we store it inside Qdrant?

**No.**

Correct location:

```text
Amazon S3 / Azure Blob Storage
```

---

# 3. Enterprise Architecture

```text
                         User
                           │
                           ▼
       Azure Front Door / Route53 + CloudFront
                           │
                           ▼
Azure API Management / Amazon API Gateway
                           │
                           ▼
Azure App Gateway / AWS ALB
                           │
                           ▼
FastAPI (Container Apps) / ECS Fargate
                           │
        ┌──────────────────┼─────────────────────┐
        ▼                  ▼                     ▼
 Amazon S3 / Blob      PostgreSQL            LangGraph
        │
        ▼
 OCR (Textract / Document Intelligence)
        │
        ▼
 Chunking
        │
        ▼
 Embeddings
        │
        ▼
 Qdrant / OpenSearch / Azure AI Search
        │
        ▼
 Azure OpenAI / AWS Bedrock
```

---

# 4. Document Upload Flow

```text
User Uploads PDF

↓

FastAPI

↓

Amazon S3 / Azure Blob

↓

Store File

↓

Return File URL
```

---

# 5. RAG Processing Flow

After upload

```text
S3 / Blob

↓

OCR

↓

Text

↓

Chunk

↓

Embedding

↓

Vector DB

↓

Metadata

↓

PostgreSQL
```

Notice

The PDF never goes into

Qdrant

or

PostgreSQL.

---

# 6. Why Not Store PDFs in PostgreSQL?

Wrong

```text
PostgreSQL

↓

PDF
```

Problems

❌ Slow

❌ Large Database

❌ Expensive

❌ Backup Size

---

Correct

```text
Amazon S3

↓

PDF

↓

PostgreSQL

↓

Metadata
```

---

# 7. What is Stored?

### Amazon S3 / Blob

- PDF
- Word
- Images
- Audio
- Video

---

### PostgreSQL

- File Name
- Owner
- Upload Date
- S3 Path
- Status

---

### Vector DB

- Embeddings

---

# 8. S3 Bucket Structure

Example

```text
company-documents/

    hr/

        leave_policy.pdf

    payroll/

        salary_policy.pdf

    healthcare/

        claims.pdf
```

---

# 9. Python Example (AWS S3)

Install

```bash
pip install boto3
```

Upload

```python
import boto3

s3 = boto3.client("s3")

s3.upload_file(
    "leave_policy.pdf",
    "company-documents",
    "hr/leave_policy.pdf"
)
```

---

Download

```python
s3.download_file(
    "company-documents",
    "hr/leave_policy.pdf",
    "leave_policy.pdf"
)
```

---

# 10. Python Example (Azure Blob)

```python
from azure.storage.blob import BlobServiceClient

blob = BlobServiceClient.from_connection_string(
    CONNECTION_STRING
)
```

Upload

```python
container = blob.get_container_client("documents")

with open("leave_policy.pdf", "rb") as data:
    container.upload_blob(
        "leave_policy.pdf",
        data
    )
```

---

# 11. S3 Security

Never

```python
AWS_SECRET = "abc"
```

Use

AWS Secrets Manager

---

Azure

↓

Key Vault

---

# 12. Versioning

Suppose

User uploads

```text
policy.pdf
```

Tomorrow

Updated

```text
policy.pdf
```

S3 Versioning

↓

Keeps both versions.

Very important.

---

# 13. Lifecycle Policies

Suppose

Old files

after

365 days.

Automatically

↓

Archive

↓

Delete

Saves cost.

---

# 14. Encryption

AWS

SSE-S3

or

KMS

Azure

Storage Encryption

Always encrypt documents.

---

# 15. Presigned URL

Instead of

Downloading through FastAPI

Generate

```text
Temporary Secure URL
```

User

↓

Downloads

directly

from S3.

Reduces server load.

---

# 16. Best Practices

✅ Store only files

✅ Store metadata separately

✅ Enable Versioning

✅ Enable Encryption

✅ Use Lifecycle Rules

✅ Use Presigned URLs

✅ Organize folders

---

# 17. Common Mistakes

❌ Store PDF in PostgreSQL

❌ Store PDF in Redis

❌ Store PDF in Vector DB

❌ Public bucket

❌ Hardcoded AWS Keys

---

# 18. Interview Questions

### Q1. Why S3?

Store unstructured files reliably and cost-effectively.

---

### Q2. Why not PostgreSQL?

Relational databases are not optimized for large binary files.

---

### Q3. Why not Vector DB?

Vector databases store embeddings, not documents.

---

### Q4. What do you store in PostgreSQL?

Metadata only.

---

### Q5. Why Versioning?

Recover previous document versions.

---

### Q6. What happens after upload?

```text
Upload

↓

S3

↓

OCR

↓

Chunk

↓

Embedding

↓

Vector DB

↓

Metadata

↓

PostgreSQL
```

---

# 19. Scenario-Based Question

### Interviewer

> User uploads a 200 MB PDF.

What happens?

Expected Answer

1. FastAPI uploads the file to Amazon S3 (or Azure Blob Storage).
2. Store metadata (filename, owner, storage path, upload status) in PostgreSQL.
3. Trigger asynchronous processing (SQS/Lambda, Celery, Azure Functions, etc.).
4. OCR if required (Textract/Document Intelligence).
5. Chunk the extracted text.
6. Generate embeddings.
7. Store embeddings in Qdrant/OpenSearch/Azure AI Search.
8. Update document processing status in PostgreSQL.

The upload API should return immediately instead of waiting for embedding generation.

---

# 20. S3 vs PostgreSQL vs Redis vs Qdrant

| Component | Stores |
|------------|--------|
| Amazon S3 / Azure Blob | Files (PDF, Images, Audio) |
| PostgreSQL | Metadata, Users, Chat History |
| Redis | Cache, Sessions |
| Qdrant / OpenSearch | Embeddings |

---

# 21. Complete Enterprise Flow

```text
User
 │
 ▼
FastAPI
 │
 ▼
Amazon S3 / Azure Blob
 │
 ▼
Metadata → PostgreSQL
 │
 ▼
Background Worker
 │
 ▼
OCR
 │
 ▼
Chunking
 │
 ▼
Embedding Model
 │
 ▼
Qdrant / OpenSearch / Azure AI Search
 │
 ▼
Ready for RAG
```

---

# 22. EPAM Senior Answer (2 Minutes)

> "In an enterprise AI application, Amazon S3 or Azure Blob Storage serves as the central repository for unstructured content such as PDFs, Word documents, images, and audio files. When a user uploads a document, FastAPI stores it in object storage and records only the metadata—such as filename, owner, storage path, and processing status—in PostgreSQL. The upload then triggers an asynchronous ingestion pipeline that performs OCR if necessary, chunks the extracted text, generates embeddings, and indexes them into Qdrant, Amazon OpenSearch, or Azure AI Search. The original document remains in object storage and is never stored in the vector database or Redis. I also enable versioning, encryption, lifecycle policies, and use presigned URLs for secure document access. This architecture is scalable, secure, and well suited for production RAG systems."